# Clustering: Wholesale Customers

**Problem statement:** Segment customers using six annual-spending features. `Channel` and `Region` are used only after fitting for interpretation.

## 1. Setup and reproducibility

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

RANDOM_STATE=42; 
np.random.seed(RANDOM_STATE); 
sns.set_theme(style="whitegrid",palette="colorblind")

FEATURES=["Fresh","Milk","Grocery","Frozen","Detergents_Paper","Delicassen"]

def metrics(x,y): 
   return {"Silhouette Score":silhouette_score(x,y), "Davies-Bouldin Index": davies_bouldin_score(x,y), "Calinski-Harabasz Index": calinski_harabasz_score(x,y)}

def pca_plot(x,y,title):
 z = PCA(2,random_state=RANDOM_STATE).fit_transform(x); 
 q=pd.DataFrame({"PC1":z[:,0],"PC2":z[:,1],"Cluster":y.astype(str)})
 plt.figure(figsize=(8,6)); 
 sns.scatterplot(data=q,x="PC1",y="PC2",hue="Cluster",palette="tab10",s=55); 
 plt.title(title); 
 plt.xlabel("Principal component 1"); 
 plt.ylabel("Principal component 2"); 
 plt.legend(title="Cluster"); 
 plt.tight_layout(); 
 plt.show()

## 2. Dataset loading and audit

In [2]:
data_path = Path("data/clustering.csv")
if not data_path.exists(): 
    data_path = Path("../data/clustering.csv")
df = pd.read_csv(data_path) 
display(df.head()) 
print(f"Dataset shape: {df.shape}")
audit = pd.DataFrame({"dtype":df.dtypes.astype(str),"missing_count":df.isna().sum(),"missing_percent":df.isna().mean().mul(100).round(2),"unique_values":df.nunique()})
display(audit) 
print("Duplicate rows:",df.duplicated().sum())

,Channel,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Region
0,2,12669,9656,7561,214,2674,1338,3
1,2,7057,9810,9568,1762,3293,1776,3
2,2,6353,8808,7684,2405,3516,7844,3
3,1,13265,1196,4221,6404,507,1788,3
4,2,22615,5410,7198,3915,1777,5185,3


Dataset shape: (440, 8)


,dtype,missing_count,missing_percent,unique_values
Channel,int64,0,0.0,2
Fresh,int64,0,0.0,433
Milk,int64,0,0.0,421
Grocery,int64,0,0.0,430
Frozen,int64,0,0.0,426
Detergents_Paper,int64,0,0.0,417
Delicassen,int64,0,0.0,403
Region,int64,0,0.0,3


Duplicate rows: 0


**Observation:** There are 440 customers, no missing values, and no duplicate rows. The six spending columns are the only fitting inputs; `Channel` and `Region` remain reference fields.